# Fan-out join · **00 — Bronze**

I download the three NHS England files for August 2026, unmodified, and
record the SHA-256 of each in `data/bronze/<slug>/metadata.json`. Those
metadata files are in git; the payloads are not.

This is not new ingestion. The URLs are the ones my August 2026
verification pipeline already gated (51 of 51 checks), and I read nothing
from anywhere else. If a file is already in Bronze I keep it and only
re-hash it.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import BRONZE_DIR, download, sha256_of, write_metadata  # noqa: E402
from sources import SOURCES  # noqa: E402

In [2]:
for src in SOURCES:
    folder = BRONZE_DIR / src["slug"]
    folder.mkdir(parents=True, exist_ok=True)
    files = []
    for name, url, note in src["files"]:
        dest = folder / name
        if not dest.exists():
            download(url, dest)
        files.append({"name": name, "url": url, "note": note,
                      "bytes": dest.stat().st_size, "sha256": sha256_of(dest)})
        print(f"{src['slug']:<28} {name:<58} {dest.stat().st_size:>10,} bytes")
    write_metadata(folder, **{k: v for k, v in src.items() if k != "files"}, files=files)

nhse_rtt_2026-08             Full-CSV-data-file-Aug26-ZIP-3M-vjgRdG.zip                  3,440,872 bytes
nhse_rtt_published_2026-08   Aug26-RTT-statistical-press-notice-PDF-389K-vjgRdG.pdf        398,227 bytes
nhse_rtt_published_2026-08   Incomplete-Provider-Aug26-XLSX-9M-vjgRdG.xlsx              10,183,411 bytes


A quick sanity check that I have real files, not an HTML challenge page:
NHS England's statistics pages sit behind a bot check, and a blocked
download can come back as a few hundred bytes of HTML with a 200.

In [3]:
for src in SOURCES:
    meta = json.loads((BRONZE_DIR / src["slug"] / "metadata.json").read_text())
    for f in meta["files"]:
        head = (BRONZE_DIR / src["slug"] / f["name"]).read_bytes()[:4]
        expected = {".zip": b"PK\x03\x04", ".xlsx": b"PK\x03\x04", ".pdf": b"%PDF"}[Path(f["name"]).suffix]
        assert head == expected, f"{f['name']} is not a real {Path(f['name']).suffix} file: {head!r}"
print("all three payloads have the right file signature")

all three payloads have the right file signature
